In [ ]:
import sys
print(sys.executable)

必要なライブラリとモジュールのインストール

In [ ]:
#セル1
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
# ▼修正箇所: Matern をここに追加しています
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import tkinter as tk
from tkinter import filedialog
import warnings
import seaborn as sns

# 警告を非表示
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- フォント設定 ---
font_file_path = r"C:\Users\0uh2j\OneDrive\Desktop\IPAexfont00401\ipaexg.ttf"
if os.path.exists(font_file_path):
    try:
        fm.fontManager.addfont(font_file_path)
        font_prop = fm.FontProperties(fname=font_file_path)
        plt.rcParams['font.family'] = font_prop.get_name()
        print(f"フォント適用: {font_prop.get_name()}")
    except Exception as e:
        print(f"フォント設定エラー: {e}")
else:
    print("デフォルトフォントを使用します。")

学習データファイル(CSVファイル)選択

In [ ]:
#セル2
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()

# ▼追加: rootウィンドウを常に最前面に設定する
root.attributes('-topmost', True)

print("分析するCSVファイルを選択してください...")

# parent=root を追加することで、ダイアログも最前面の設定を引き継ぐ
csv_file_path = filedialog.askopenfilename(
    parent=root,
    title="CSVファイルを選択してください",
    filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
)

data_loaded = False

データ読み込みと前処理

In [ ]:
#セル3

if csv_file_path:
    try:
        df = pd.read_csv(csv_file_path)
        
        # 列名の定義
        col_temp = '流動中樹脂温度T(℃)'
        col_pres = '流動中樹脂圧力P(Pa)'
        col_sr = 'せん断速度γ(1/s)'
        col_visc = '樹脂粘度η(Pa・s)'
        
        required_columns = [col_temp, col_pres, col_sr, col_visc]
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"必要な列が不足しています: {required_columns}")

        # 前処理 (単位変換と対数変換)
        df['流動中樹脂温度T(K)'] = df[col_temp] + 273.15
        
        # ログ計算用の一時変数
        # ※インデックスを保持するためにdfに新しい列として追加する方法もありますが、
        # ここではシンプルに配列として処理し、インデックス管理は後で行います。
        log_T = np.log(df['流動中樹脂温度T(K)'].values)
        log_P = np.log(df[col_pres].values)
        log_SR = np.log(df[col_sr].values)
        log_VISC = np.log(df[col_visc].values)
        
        # 入力データ X と目的変数 y の作成
        X_raw = np.stack([log_T, log_P, log_SR], axis=1)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)
        
        y_mean = log_VISC.mean()
        y_centered = log_VISC - y_mean
        
        data_loaded = True
        print(f"データ読み込み完了: 全 {len(X_scaled)} 件")
        
    except Exception as e:
        print(f"エラーが発生しました: {e}")

学習データとテストデータの分割設定(公差検証法)

In [ ]:
# セル4: 【重要】学習データとテストデータの分割設定 (範囲指定対応版)
if data_loaded:
    print("--- データの分割設定 ---")

    # ==========================================================
    # ▼▼▼ 設定エリア：ここで分割方法を決定してください ▼▼▼
    
    # モード選択: "random" または "condition"
    split_mode = "condition" 
    
    # 【設定A】 "random" の場合の設定
    test_ratio = 0.2  # テストデータの割合 (0.2 = 20%)
    
    # 【設定B】 "condition" の場合の設定 (範囲指定)
    # 指定した「下限値」から「上限値」の範囲内にあるデータをテスト用(検証用)にします。
    # それ以外のデータ(範囲外)を使って学習を行います。
    # 例: 温度が 200℃ 〜 210℃ のデータをテスト用にくり抜く場合
    
    target_column = 'せん断速度γ(1/s)' # 条件を判断する列名
    
    test_range_min = 500   # テストデータにする範囲の下限
    test_range_max = 1000   # テストデータにする範囲の上限
    
    # ▲▲▲ 設定エリア終了 ▲▲▲
    # ==========================================================

    indices = np.arange(len(X_scaled))

    if split_mode == "random":
        # ランダム分割
        X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
            X_scaled, y_centered, indices, test_size=test_ratio, random_state=42
        )
        print(f"【ランダム分割モード】 全体の {test_ratio*100}% をテストデータにしました。")

    elif split_mode == "condition":
        # 条件指定分割 (範囲指定)
        # ターゲット列の値を取得
        target_values = df[target_column].values
        
        # 範囲内 (min <= x <= max) をテストデータとするマスクを作成
        test_mask = (target_values >= test_range_min) & (target_values <= test_range_max)
        
        condition_str = f"{test_range_min} <= {target_column} <= {test_range_max}"
            
        idx_test = indices[test_mask]
        idx_train = indices[~test_mask] # maskの反転(Not) = 学習データ
        
        # エラーチェック
        if len(idx_test) == 0:
            print(f"【警告】指定された範囲({condition_str})にデータが1件もありません。")
            print("設定を見直すか、範囲を広げてください。")
        elif len(idx_train) == 0:
            print(f"【警告】全てのデータがテスト範囲に含まれてしまいました。学習できるデータがありません。")
            print("範囲を狭めてください。")
        else:
            X_train = X_scaled[idx_train]
            y_train = y_centered[idx_train]
            X_test = X_scaled[idx_test]
            y_test = y_centered[idx_test]
            print(f"【条件指定モード(範囲)】")
            print(f"条件: 「{condition_str}」 のデータをテスト(検証)用にしました。")
            print(f"それ以外のデータを学習に使用します。")

    # データ数の表示
    if 'X_train' in locals() and 'X_test' in locals():
        print(f"  - 学習用データ数: {len(X_train)} 件")
        print(f"  - テスト用データ数: {len(X_test)} 件")
        # 学習実行フラグ
        ready_to_train = len(X_train) > 0 and len(X_test) > 0
    else:
        ready_to_train = False

機械学習モデル構築と学習

In [ ]:
#セル5 学習モデル構築

from scipy.optimize import minimize
import matplotlib.pyplot as plt

# 学習過程を記録するためのリスト
training_loss_history = []

# カスタムオプティマイザ
def custom_optimizer(obj_func, initial_theta, bounds):
    global training_loss_history
    training_loss_history = [] 
    print(f"  > 最適化計算中", end="")
    def callback(xk):
        val, _ = obj_func(xk)
        training_loss_history.append(val)
        print(".", end="", flush=True)
    res = minimize(obj_func, initial_theta, bounds=bounds, method="L-BFGS-B", jac=True, callback=callback)
    print(" 完了")
    return res.x, res.fun

if data_loaded and ready_to_train:
    print("\n--- 学習プロセス開始 (ARD Kernel) ---")
    
    N_MAX_TRAIN = 2000
    if len(X_train) > N_MAX_TRAIN:
        print(f"※学習データ({len(X_train)})が多いため、ランダムに{N_MAX_TRAIN}件を使用してモデル構築します。")
        fit_indices = np.random.choice(len(X_train), N_MAX_TRAIN, replace=False)
        X_train_fit = X_train[fit_indices]
        y_train_fit = y_train[fit_indices]
    else:
        X_train_fit = X_train
        y_train_fit = y_train

    
    # 【パターン1】 ARD Matérn 5/2 カーネル (おすすめ：滑らかさと局所変化のバランス)
    kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0, 1.0, 1.0], length_scale_bounds=(1e-2, 1e2), nu=2.5) + \
             WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-3, 1e1))
    
    gp_model = GaussianProcessRegressor(
        kernel=kernel, 
        n_restarts_optimizer=5,
        optimizer=custom_optimizer,
        random_state=42
    )
    
    gp_model.fit(X_train_fit, y_train_fit)
    
    print("--- 学習完了 ---")
    print(f"獲得したカーネル: {gp_model.kernel_}")
    learning_completed = True

対数周辺尤度値

In [ ]:
# セル6: 学習結果の尤度（モデルの適合度）を確認
if learning_completed:
    print("--- 学習の確信度チェック ---")
    
    # 対数周辺尤度 (Log Marginal Likelihood)
    # 値が大きいほど「データに適合している」ことを示します
    lml_value = gp_model.log_marginal_likelihood(gp_model.kernel_.theta)
    
    print(f"対数周辺尤度 (LML): {lml_value:.4f}")
    print("※ この値自体に絶対的な基準はありませんが、")
    print("   他のカーネル条件と比較する際や、学習が失敗していないかの目安になります。")
    print("   （極端に小さな値、例えば -数万 などになっている場合は学習失敗の可能性があります）")

損失関数の推移

In [ ]:
# セル7：損失関数の推移グラフ
import matplotlib.pyplot as plt

if learning_completed and 'training_loss_history' in globals() and len(training_loss_history) > 0:
    plt.figure(figsize=(10, 6))
    
    # 損失の推移をプロット
    plt.plot(training_loss_history, marker='o', linestyle='-', color='blue', markersize=4)
    
    plt.title("学習プロセスにおける損失関数の推移\n(負の対数周辺尤度)", fontsize=14)
    plt.xlabel("計算ステップ数 (Iteration)", fontsize=12)
    plt.ylabel("損失 (値が小さいほど良い)", fontsize=12)
    plt.grid(True, which='both', linestyle='--', alpha=0.7)
    
    # 最後の値を表示
    final_loss = training_loss_history[-1]
    plt.text(len(training_loss_history)-1, final_loss, f"{final_loss:.2f}", 
             ha='left', va='bottom', color='red', fontweight='bold')

    plt.show()
    
    print(f"最終的な損失値: {final_loss:.4f}")
    if len(training_loss_history) < 5:
        print("※ ステップ数が少ないのは、すぐに最適解が見つかったためです。")
    elif training_loss_history[-1] < training_loss_history[0]:
        print(">> 判定: 損失が減少しており、学習は正常に進行しました。")
    else:
        print(">> 判定: 損失があまり変化していません。初期値が良かったか、学習が停滞した可能性があります。")

else:
    print("まだ学習が行われていないか、履歴が記録されていません。セル5を再実行してください。")

説明変数ごとの特徴スケール

In [ ]:
# セル7.5：説明変数ごとの特徴スケールの出力

def plot_gp_feature_importance(gpr_model, feature_names):
    """
    GPRモデルからlength_scaleを取り出し、重要度をランキング表示する
    """
    # 1. 学習済みのカーネルから length_scale を取得
    # 複合カーネル（Constant * RBF + WhiteKernel など）に対応させる処理
    ls = None
    kernel = gpr_model.kernel_
    
    # カーネルのパラメータを全て取得して探索
    params = kernel.get_params()
    for key, value in params.items():
        # 'length_scale'を含み、かつ数値(配列)であるものを探す
        if 'length_scale' in key and not isinstance(value, str):
            ls = value
            break
            
    if ls is None:
        print("エラー: length_scale が見つかりませんでした。")
        return None

    # 配列になっていない（ARDではない）場合は警告
    if np.ndim(ls) == 0:
        print("警告: カーネルが等方性(スカラー)です。特徴量ごとの重要度差はありません。")
        # 便宜上、すべて同じ値として処理
        ls = np.array([ls] * len(feature_names))

    # 2. 重要度を計算 (1 / length_scale)
    # RBFカーネルでは、length_scaleが「小さい」ほど、その次元の変化に敏感（＝重要）
    importance = 1.0 / ls

    # 3. データフレームにまとめてソート
    df_importance = pd.DataFrame({
        'Feature': feature_names,
        'Length_Scale': ls,
        'Importance': importance
    }).sort_values(by='Importance', ascending=False)

    # 4. グラフ表示
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=df_importance, palette='viridis')
    plt.title('Feature Importance based on GP Length Scales (ARD)')
    plt.xlabel('Importance (Sensitivity = 1 / length_scale)')
    plt.ylabel('Features')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    return df_importance

# --- 実行部分 ---
if 'gp_model' in globals() and learning_completed:
    print("--- 特徴量の重要度分析 ---")
    
    # セル3で作成したデータの順番に合わせて名前を定義します
    # 順番: log_T, log_P, log_SR
    feature_labels = ['Log(温度T)', 'Log(圧力P)', 'Log(せん断速度γ)']
    
    # 関数を実行（ここで gp_model を渡します）
    result_df = plot_gp_feature_importance(gp_model, feature_labels)
    
    print("\n【分析結果】")
    print(result_df)
    print("\n※ Importanceが大きいほど、その変数が粘度に与える影響が大きいことを示唆します。")

else:
    print("エラー: 学習済みモデル(gp_model)が見つかりません。セル5を実行してください。")

選択した変数の共分散行列の可視化(ヒートマップ)
（粘度予測モデルの数値的な結論）

In [ ]:
# セル８　共分散行列（カーネル行列）のヒートマップ可視化
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

if learning_completed:
    print("--- 共分散行列の可視化（並び替え比較） ---")
    
    sort_targets = [
        ("せん断速度 (説明変数)", 'X', 2),
        ("流動中樹脂温度 (説明変数)", 'X', 0),
        ("流動中樹脂圧力 (説明変数)", 'X', 1),
        ("樹脂粘度 (目的変数)", 'y', None) 
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    
    for i, (name, source, idx) in enumerate(sort_targets):
        ax = axes[i]
        
        # 1. 並べ替えの基準となる値を取得
        sort_values = X_train_fit[:, idx] if source == 'X' else y_train_fit
            
        # 2. 並べ替えインデックスを取得
        sort_indices = np.argsort(sort_values)
        
        # 3. データを並べ替え
        X_sorted = X_train_fit[sort_indices]
        
        # 4. 共分散行列を計算
        K_matrix = gp_model.kernel_(X_sorted)
        
        # --- 修正箇所: 目盛りの間隔設定 ---
        # データ数に応じて目盛りの表示間隔を自動調整（例：全体の1/10程度の間隔）
        tick_spacing = max(1, len(K_matrix) // 10) 

        # 5. ヒートマップ描画
        # xticklabels/yticklabels に数値を指定すると、その間隔で目盛り数字が表示されます
        sns.heatmap(K_matrix, cmap="viridis", square=True,
                    xticklabels=tick_spacing, yticklabels=tick_spacing,
                    cbar=True, cbar_kws={'label': '相関の強さ'}, ax=ax)
        
        # --- 修正箇所: 目盛りとラベルの見た目調整 ---
        ax.tick_params(labelsize=9, left=True, bottom=True) # 目盛り線を可視化
        ax.set_title(f"並び順: {name}", fontsize=14, pad=20)
        ax.set_xlabel("データ点 ID ", fontsize=11)
        ax.set_ylabel("データ点 ID ", fontsize=11)

    plt.tight_layout()
    plt.show()

予測分布グラフ(粘度に対するそれぞれの説明変数の依存性グラフ)

In [ ]:
#セル9　目的変数に対する各説明変数の依存性グラフ

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

if learning_completed:
    # === 2. 依存性プロット (傾向確認・個別出力版) ===
    print("\n--- 変数ごとの依存性プロット ---")
    
    # 【設定項目】グラフサイズをここで一括設定できます (幅, 高さ)
    common_figsize = (12, 6)
    
    # 表示設定
    plot_configs = [
        {'name': 'せん断速度γ',    'unit': '(1/s)', 'data_idx': 2},
        {'name': '流動中樹脂温度T',  'unit': '(℃)',   'data_idx': 0},
        {'name': '流動中樹脂圧力P',  'unit': '(Pa)',  'data_idx': 1}
    ]
    
    n_grid = 100
    
    # --- 色分け設定 (せん断速度に基づく) ---
    shear_rates_log_all = X_raw[:, 2] 
    norm = plt.Normalize(vmin=shear_rates_log_all.min(), vmax=shear_rates_log_all.max())
    cmap = 'viridis' 
    
    for i, config in enumerate(plot_configs):
        # 1つずつ個別のFigureを作成
        fig, ax = plt.subplots(figsize=common_figsize)
        
        d_idx = config['data_idx']
        var_name = config['name']
        
        # 予測用グリッド作成
        x_min, x_max = X_scaled[:, d_idx].min(), X_scaled[:, d_idx].max()
        padding = (x_max - x_min) * 0.1
        x_grid = np.linspace(x_min - padding, x_max + padding, n_grid)
        
        X_new_scaled = np.zeros((n_grid, 3))
        X_new_scaled[:, d_idx] = x_grid
        
        # 予測
        y_pred, y_std = gp_model.predict(X_new_scaled, return_std=True)
        
        # 逆変換
        visc_mean = np.exp(y_pred + y_mean)
        visc_lower = np.exp(y_pred - 3.0*y_std + y_mean)
        visc_upper = np.exp(y_pred + 3.0*y_std + y_mean)
        
        x_grid_real = np.exp(x_grid * scaler.scale_[d_idx] + scaler.mean_[d_idx])
        if d_idx == 0: x_grid_real -= 273.15
            
        # 学習データのプロット 
        idx_sample = np.random.choice(len(X_raw), size=min(2000, len(X_raw)), replace=False)
        x_raw_sample = np.exp(X_raw[idx_sample, d_idx])
        y_raw_sample = np.exp(y_centered[idx_sample] + y_mean)
        c_sample = X_raw[idx_sample, 2] # logせん断速度
        
        if d_idx == 0: x_raw_sample -= 273.15
            
        # 散布図プロット
        sc = ax.scatter(x_raw_sample, y_raw_sample, c=c_sample, cmap=cmap, norm=norm,
                        alpha=0.5, s=15, label='全データ(色はせん断速度)')
        
        ax.plot(x_grid_real, visc_mean, color='red', linewidth=2, label='予測平均')
        ax.fill_between(x_grid_real, visc_lower, visc_upper, color='red', alpha=0.2, label='95%信頼区間')
        
        ax.set_title(f"{var_name} 依存性", fontsize=14)
        ax.set_xlabel(f"{var_name} {config['unit']}", fontsize=12)
        ax.set_ylabel("樹脂粘度 (Pa・s)", fontsize=12)
        ax.set_yscale('log')
        
        # せん断速度のグラフの場合の設定（10の整数乗のみ表示）
        if d_idx == 2: 
            ax.set_xscale('log')
            ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0))
            ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
            ax.xaxis.set_minor_formatter(ticker.NullFormatter())
            ax.tick_params(axis='x', which='major', rotation=0, labelsize=10)

        ax.grid(True, which="both", linestyle='--', alpha=0.5)
        ax.legend(loc='upper right', fontsize='small')
        
        # カラーバーを追加
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label('log(せん断速度)', fontsize=12)

        # ループ内でその都度表示
        plt.tight_layout()
        plt.show()

In [ ]:
#セル9-2 目的変数に対する各説明変数の依存性グラフ（予測線延長・完全版）

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os
from matplotlib.patches import Patch

if learning_completed and 'df' in locals() and 'キャビティ厚み' in df.columns:
    print("\n--- 依存性プロットの作成 (予測線延長・凡例左揃え) ---")
    
    base_dir = os.path.dirname(csv_file_path) if 'csv_file_path' in globals() and csv_file_path else "."
    
    # ==========================================
    # ▼▼ 設定エリア（グラフの見た目・文字サイズ） ▼▼
    # ==========================================
    common_figsize = (15, 10)
    report_dpi = 600 
    
    axis_label_fontsize = 24  # 軸名の文字サイズ
    legend_fontsize = 24      # 凡例内の文字サイズ
    title_fontsize = 24       # タイトルの文字サイズ
    tick_labelsize = 28       # 軸の目盛り数字のサイズ
    
    # ▼▼ プロットと凡例のマーカーサイズ設定 ▼▼
    plot_marker_size = 200    # グラフ上の散布図マーカーのサイズ（面積）
    legend_marker_size = 200  # 凡例内の散布図マーカーのサイズ（面積）
    
    show_xlabel = False
    show_ylabel = False
    
    # ==========================================
    # ▼▼ 軸の表示範囲設定（ここで数値を指定します） ▼▼
    # ==========================================
    # Y軸（樹脂粘度）の目盛り範囲設定
    y_axis_min = 1.0       # 例: 10^0
    y_axis_max = 50000.0   # 例: 10^4
    
    # X軸の目盛り範囲設定
    # 変更したい場合は、以下の x_min と x_max の「右側の数値」を書き換えてください。
    # ※対数グラフのため x_min に 0 は指定できません（1.0 などを指定してください）
    # ※自動判定にしたい場合は None と書いてください。
    plot_configs = [
        {'name': 'せん断速度', 'symbol': r'$\gamma$', 'unit': '(1/s)', 'data_idx': 2, 'x_min': 1.0, 'x_max': 200000.0},
        {'name': '流動中樹脂温度', 'symbol': r'$T$', 'unit': '(℃)', 'data_idx': 0, 'x_min': None, 'x_max': None},
        {'name': '流動中樹脂圧力', 'symbol': r'$P$', 'unit': '(Pa)', 'data_idx': 1, 'x_min': None, 'x_max': None}
    ]
    # ==========================================
    
    style_dict = {
        0.5: {'color': '#d62728', 'size': plot_marker_size, 'marker': 'o', 'zorder': 14},
        1.0: {'color': '#1f77b4', 'size': plot_marker_size, 'marker': 's', 'zorder': 13},
        1.5: {'color': "#ffd30ed9", 'size': plot_marker_size, 'marker': '^', 'zorder': 12},
        2.0: {'color': "#41ea4f", 'size': plot_marker_size, 'marker': 'D', 'zorder': 11}
    }
    default_style = {'color': 'gray', 'size': plot_marker_size / 4.6, 'marker': 'x', 'zorder': 10}

    n_grid = 100
    cavity_all = df['キャビティ厚み'].values
    
    for i, config in enumerate(plot_configs):
        fig, ax = plt.subplots(figsize=common_figsize, dpi=report_dpi)
        
        d_idx = config['data_idx']
        var_full_name = f"{config['name']} {config['symbol']}"
        
        # ▼▼ 予測データの計算 ▼▼
        if config['x_min'] is not None and config['x_max'] is not None:
            # プログラム内部の処理（ここは変更しないでください）
            real_min = config['x_min']
            real_max = config['x_max']
            if d_idx == 0: 
                real_min += 273.15
                real_max += 273.15
            scaled_min = (np.log(real_min) - scaler.mean_[d_idx]) / scaler.scale_[d_idx]
            scaled_max = (np.log(real_max) - scaler.mean_[d_idx]) / scaler.scale_[d_idx]
            x_grid = np.linspace(scaled_min, scaled_max, n_grid)
        else:
            x_min_data, x_max_data = X_scaled[:, d_idx].min(), X_scaled[:, d_idx].max()
            padding = (x_max_data - x_min_data) * 0.1
            x_grid = np.linspace(x_min_data - padding, x_max_data + padding, n_grid)
            
        X_new_scaled = np.zeros((n_grid, 3))
        X_new_scaled[:, d_idx] = x_grid
        y_pred, y_std = gp_model.predict(X_new_scaled, return_std=True)
        
        visc_mean = np.exp(y_pred + y_mean)
        visc_lower = np.exp(y_pred - 1.96*y_std + y_mean)
        visc_upper = np.exp(y_pred + 1.96*y_std + y_mean)
        x_grid_real = np.exp(x_grid * scaler.scale_[d_idx] + scaler.mean_[d_idx])
        if d_idx == 0: x_grid_real -= 273.15
            
        # 線の描画
        line_mean = ax.plot(x_grid_real, visc_mean, color='red', linewidth=2, 
                            label='予測平均値', zorder=20)
        fill_ci = ax.fill_between(x_grid_real, visc_lower, visc_upper, color='red', alpha=0.25, 
                                  label='95% 信頼区間', zorder=15)
        ax.plot(x_grid_real, visc_lower, color='red', linewidth=0.5, alpha=0.6, zorder=16)
        ax.plot(x_grid_real, visc_upper, color='red', linewidth=0.5, alpha=0.6, zorder=16)
        
        # 学習データのプロット
        idx_sample = np.random.choice(len(X_raw), size=min(2000, len(X_raw)), replace=False)
        x_raw_sample = np.exp(X_raw[idx_sample, d_idx])
        y_raw_sample = np.exp(y_centered[idx_sample] + y_mean)
        cavity_sample = cavity_all[idx_sample]
        if d_idx == 0: x_raw_sample -= 273.15
            
        unique_cavities = np.sort(np.unique(cavity_sample))
        scatter_handles = []
        
        for cav_val in unique_cavities:
            mask = (cavity_sample == cav_val)
            style = style_dict.get(cav_val, default_style)
            sc = ax.scatter(x_raw_sample[mask], y_raw_sample[mask], 
                            color=style['color'], s=style['size'], marker=style['marker'],
                            alpha=0.8, edgecolors='k', linewidths=0.5, 
                            zorder=style['zorder'])
            scatter_handles.append(sc)
        
        # 凡例の設定
        spacer_handle = Patch(color='none', label='')
        final_handles = [line_mean[0], fill_ci, spacer_handle] + scatter_handles
        final_labels = ['予測平均値', '95% 信頼区間', 'キャビティ厚み'] + [f'{cav_val} mm' for cav_val in unique_cavities]
        
        # 凡例マーカーの表示倍率を計算
        scale_factor = np.sqrt(legend_marker_size / plot_marker_size) if plot_marker_size > 0 else 1.0
        
        leg = ax.legend(final_handles, final_labels, loc='upper right', fontsize=legend_fontsize, 
                        framealpha=1.0, edgecolor='black', markerscale=scale_factor)
        
        # 目盛りの設定
        ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, labelsize=tick_labelsize, pad=8)
        
        # タイトルの設定
        if d_idx == 2: 
            ax.set_title("ガウス過程回帰による予測モデル\n（せん断速度と粘度の関係）", fontsize=title_fontsize, fontweight='bold', pad=15)
        else: 
            ax.set_title(f"樹脂粘度の {config['name']} 依存性", fontsize=title_fontsize, fontweight='bold', pad=15)
        
        # 軸名の設定
        if show_xlabel:
            ax.set_xlabel(f"{var_full_name} {config['unit']}", fontsize=axis_label_fontsize, fontweight='bold', labelpad=15)
        if show_ylabel:
            ax.set_ylabel("樹脂粘度 (Pa·s)", fontsize=axis_label_fontsize, fontweight='bold', labelpad=15)
        
        # Y軸スケール
        ax.set_yscale('log')
        if y_axis_min is not None or y_axis_max is not None:
            cur_ymin, cur_ymax = ax.get_ylim()
            ax.set_ylim(y_axis_min if y_axis_min is not None else cur_ymin, 
                        y_axis_max if y_axis_max is not None else cur_ymax)
        ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
        ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
        
        # X軸スケール
        if config['x_min'] is not None or config['x_max'] is not None:
            cur_xmin, cur_xmax = ax.get_xlim()
            ax.set_xlim(config['x_min'] if config['x_min'] is not None else cur_xmin, 
                        config['x_max'] if config['x_max'] is not None else cur_xmax)
        
        if d_idx in [1, 2]: 
            ax.set_xscale('log')
            ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
            ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
            ax.xaxis.set_minor_formatter(ticker.NullFormatter())
            ax.tick_params(axis='x', which='both', direction='in', top=True, labelsize=tick_labelsize, pad=8)

        ax.grid(True, which="major", linestyle='-', alpha=0.6, color='gray', zorder=1)
        ax.grid(True, which="minor", linestyle=':', alpha=0.4, zorder=1)
        
        plt.tight_layout()
        
        save_name = 'Shear_Rate' if d_idx == 2 else ('Temperature' if d_idx == 0 else 'Pressure')
        save_filename = f"DependencyPlot_ENG_{save_name}.png"
        plt.savefig(os.path.join(base_dir, save_filename), bbox_inches='tight', dpi=report_dpi)
        plt.show()

結果の検証（パリティプロット）

In [ ]:
#セル10 テストデータを用いた粘度予測モデルの精度検証

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os
from sklearn.metrics import r2_score, mean_squared_error

if learning_completed:
    print("\n--- 検証結果の可視化 (縦軸:予測値 / 横軸:実測値) ---")
    
    # ==========================================
    # ▼▼ 設定エリア（グラフの見た目・文字サイズ） ▼▼
    # ==========================================
    report_dpi = 600
    
    axis_label_fontsize = 28  # 軸名の文字サイズ
    legend_fontsize = 28      # 凡例内の文字サイズ
    title_fontsize = 32       # タイトルの文字サイズ
    tick_labelsize = 28       # 軸の目盛り数字のサイズ 
    
    # ▼▼ プロットと凡例のマーカーサイズ設定 ▼▼
    plot_marker_size = 15      # グラフ上のマーカー(●)のサイズ
    legend_marker_size = 25    # 凡例内のマーカー(●)のサイズ
    
    # ▼▼ エラーバーの太さ設定 ▼▼ 【今回追加した部分】
    errorbar_linewidth = 3.0   # エラーバーの縦線の太さ (元は 1.5)
    cap_thickness = 3.0        # エラーバーの上下の横線(キャップ)の太さ
    
    # ==========================================
    # ▼▼ 軸の表示範囲設定（手動で数値を指定できます） ▼▼
    # ==========================================
    # パリティプロットはX軸(実測値)とY軸(予測値)が同じ単位(粘度)なので一括指定します
    # 自動判定にする場合は None を指定してください。
    axis_min = 50       # 例: 10^0
    axis_max = 300   # 例: 10^4
    # ==========================================
    
    # === 1. データ準備 ===
    y_pred_test_centered, y_std_test = gp_model.predict(X_test, return_std=True)
    
    # --- A. 実数値（予測値の中心）への変換 ---
    y_pred_log = y_pred_test_centered + y_mean
    y_pred_real = np.exp(y_pred_log)
    y_test_real = np.exp(y_test + y_mean)
    
    # --- B. 予測幅（信頼区間）の計算 ---
    sigma_coef = 1.96
    y_pred_log_upper = y_pred_log + sigma_coef * y_std_test
    y_pred_log_lower = y_pred_log - sigma_coef * y_std_test
    
    y_pred_real_upper = np.exp(y_pred_log_upper)
    y_pred_real_lower = np.exp(y_pred_log_lower)
    
    y_err_lower = y_pred_real - y_pred_real_lower
    y_err_upper = y_pred_real_upper - y_pred_real
    y_err = [y_err_lower, y_err_upper] 
    
    # --- C. 精度スコア計算 ---
    r2 = r2_score(y_test_real, y_pred_real)
    
    # --- D. プロット描画 ---
    fig, ax = plt.subplots(figsize=(14, 10), dpi=report_dpi)
    
    label_text = '予測結果\n  ● : 予測平均\n  | : 予測幅(95%信頼区間)'
    
    # エラーバーと散布図 【太さの変数を適用】
    ax.errorbar(y_test_real, y_pred_real, yerr=y_err, 
                 fmt='o', c='green', ecolor='gray', capsize=4, alpha=0.8,
                 markersize=plot_marker_size, 
                 elinewidth=errorbar_linewidth,   # 縦線の太さ
                 capthick=cap_thickness,          # 上下の横線(キャップ)の太さ
                 label=label_text)
    
    # 軸の範囲設定の計算（手動設定がない場合は自動計算）
    if axis_min is None or axis_max is None:
        data_min = min(y_test_real.min(), y_pred_real.min())
        data_max = max(y_test_real.max(), y_pred_real.max())
        margin_val = data_min * 0.1
        final_min = axis_min if axis_min is not None else (data_min - margin_val)
        final_max = axis_max if axis_max is not None else (data_max + margin_val)
    else:
        final_min = axis_min
        final_max = axis_max

    # 理想線
    ax.plot([final_min, final_max], 
             [final_min, final_max], 
             'r--', linewidth=2.5, label='理想線 (計測値 = 予測値)')
    
    # 軸設定 (対数スケール)
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    ax.set_xlim(final_min, final_max)
    ax.set_ylim(final_min, final_max)
    
    # タイトルと軸ラベル
    ax.set_title(f"モデル予測値とテストデータの実測値の比較\n決定係数 R2: {r2:.4f}", 
                 fontsize=title_fontsize, fontweight='bold', pad=15)
    ax.set_xlabel("計測値 (Pa・s) [実測データ]", fontsize=axis_label_fontsize, fontweight='bold', labelpad=15)
    ax.set_ylabel("予測モデル算出値 (Pa・s)", fontsize=axis_label_fontsize, fontweight='bold', labelpad=15)
    
    # 目盛りと軸の装飾
    ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, labelsize=tick_labelsize, pad=8)
    
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
    ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, subs=(1.0,)))
    ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    
    # 凡例とグリッドの設定
    leg = ax.legend(loc='upper left', fontsize=legend_fontsize, framealpha=1.0, edgecolor='black')
    
    # 凡例内のマーカーサイズを個別に上書き適用
    for legobj in leg.legend_handles:
        if hasattr(legobj, 'has_yerr') or hasattr(legobj, 'has_xerr'):
            legobj[0].set_markersize(legend_marker_size)
            # 凡例内のエラーバーの太さも反映させたい場合は以下を有効化します
            # legobj[1][0].set_linewidth(errorbar_linewidth)
            # legobj[2][0].set_linewidth(cap_thickness)
    
    ax.grid(True, which="major", linestyle='-', alpha=0.6, color='gray')
    ax.grid(True, which="minor", linestyle=':', alpha=0.4)
    
    plt.tight_layout()
    
    # --- 画像の保存 ---
    base_dir = os.path.dirname(csv_file_path) if 'csv_file_path' in globals() and csv_file_path else "."
    save_filename = "ParityPlot_HighRes.png"
    plt.savefig(os.path.join(base_dir, save_filename), bbox_inches='tight', dpi=report_dpi)
    
    plt.show()
    
    print(f"決定係数(R2): {r2:.4f}")
    print(">> 解説: エラーバー（縦棒）は、ガウス過程モデルが算出した「確率的な予測範囲」です。")
    print("        学習データが近くにある領域では棒が短く、未知の領域では棒が長くなります。")
    print(f">> 高解像度画像({save_filename})をデータファイルと同じフォルダに保存しました。")

In [ ]:
# セル11　学習済みモデルの保存
import os
import joblib

if 'gp_model' in locals() and learning_completed:
    # 1. セル2で選択したCSVファイルの「ディレクトリ（フォルダの場所）」を取得
    base_dir = os.path.dirname(csv_file_path)
    
    # 2. 保存するファイル名を決定
    file_name = 'viscosity_model.pkl'
    
    # 3. パスを結合して、保存先のフルパスを作成
    save_path = os.path.join(base_dir, file_name)
    
    # 4. モデルの保存
    joblib.dump(gp_model, save_path)
    
    print("--- モデル保存完了 ---")
    print(f"保存ファイル名: {file_name}")
    print(f"保存先フォルダ: {base_dir}")
    print(f"フルパス: {save_path}")

else:
    print("エラー: モデルが学習されていないため、保存できません。")

樹脂粘度予測モデル(表示ひとつの変数を固定した三次元グラフ)の作成準備：固定する値をその変数における平均値に設定

In [ ]:
# セル12　3次元グラフの準備
import numpy as np

if 'X_raw' in locals() and 'df' in locals():
    print("--- 3Dグラフ描画の準備 ---")
    
    # 元データの最小値・最大値・中央値を取得（グラフの範囲と固定値に使用）
    # X_rawの列順序: 0:温度(log), 1:圧力(log), 2:せん断速度(log)
    
    # --- 温度 (Temperature) ---
    temp_min = np.exp(X_raw[:, 0].min()) - 273.15
    temp_max = np.exp(X_raw[:, 0].max()) - 273.15
    temp_med = np.exp(np.median(X_raw[:, 0])) - 273.15
    
    # --- 圧力 (Pressure) ---
    pres_min = np.exp(X_raw[:, 1].min())
    pres_max = np.exp(X_raw[:, 1].max())
    pres_med = np.exp(np.median(X_raw[:, 1]))
    
    # --- せん断速度 (Shear Rate) ---
    sr_min = np.exp(X_raw[:, 2].min())
    sr_max = np.exp(X_raw[:, 2].max())
    sr_med = np.exp(np.median(X_raw[:, 2]))

    print(f"【算出された固定値のデフォルト(データの中央値)】")
    print(f"  - 温度固定時の値: {temp_med:.2f} ℃")
    print(f"  - 圧力固定時の値: {pres_med:.2e} Pa")
    print(f"  - せん断速度固定時の値: {sr_med:.2f} 1/s")
    print("準備完了。以下のセル10～12でグラフを描画できます。")
else:
    print("エラー: データを読み込み、学習を完了させてから実行してください。")

樹脂粘度vsせん断速度and流動中温度(モデルの部分像)

In [ ]:
# セル13

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import plotly.graph_objects as go
import os

# === 設定エリア：固定する値を変更したい場合はここを書き換えてください ===
FIXED_PRESSURE = pres_med  # デフォルトは中央値 (単位: Pa)
# ===================================================================

if learning_completed:
    # 1. グリッドデータの作成
    n_grid = 50
    sr_mesh = np.geomspace(sr_min, sr_max, n_grid)
    temp_mesh = np.linspace(temp_min, temp_max, n_grid)
    
    X_grid, Y_grid = np.meshgrid(sr_mesh, temp_mesh) # X:SR, Y:Temp
    
    # 2. モデル入力用データへの変換
    log_T_in = np.log(Y_grid.ravel() + 273.15)
    log_P_in = np.full_like(log_T_in, np.log(FIXED_PRESSURE)) # 固定
    log_SR_in = np.log(X_grid.ravel())
    
    X_input_raw = np.stack([log_T_in, log_P_in, log_SR_in], axis=1)
    X_input_scaled = scaler.transform(X_input_raw)
    
    # 3. 予測
    y_pred_scaled = gp_model.predict(X_input_scaled)
    visc_pred = np.exp(y_pred_scaled + y_mean).reshape(X_grid.shape)
    
    # --- A. 既存のMatplotlibによる描画 (画面表示用) ---
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(np.log10(X_grid), Y_grid, np.log10(visc_pred), 
                           cmap=cm.viridis, edgecolor='none', alpha=0.9)
    
    ax.set_title(f"3D Map: せん断速度 vs 温度 (圧力={FIXED_PRESSURE:.2e} Pa 固定)", fontsize=14)
    ax.set_xlabel('log10(せん断速度 [1/s])', fontsize=12)
    ax.set_ylabel('温度 [℃]', fontsize=12)
    ax.set_zlabel('log10(樹脂粘度 [Pa·s])', fontsize=12)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='log10(粘度)')
    plt.show()

    # --- B. PlotlyによるHTMLファイル保存 (ブラウザ操作用) ---
    print("\n>> HTML形式の3Dグラフを作成・保存中...")
    
    # 保存先のパス設定
    base_dir = os.path.dirname(csv_file_path)
    file_name = "3D_Graph_SR_vs_Temp.html"
    save_path = os.path.join(base_dir, file_name)
    
    # Plotlyグラフ作成
    fig_html = go.Figure(data=[go.Surface(
        z=np.log10(visc_pred), 
        x=np.log10(X_grid), 
        y=Y_grid,
        colorscale='Viridis',
        colorbar=dict(title='log10(粘度)')
    )])
    
    fig_html.update_layout(
        title=f"3D Map: せん断速度 vs 温度 (圧力={FIXED_PRESSURE:.2e} Pa 固定)",
        scene=dict(
            xaxis_title='log10(せん断速度 [1/s])',
            yaxis_title='温度 [℃]',
            zaxis_title='log10(樹脂粘度 [Pa·s])'
        ),
        width=1000, height=800
    )
    
    # ファイル保存
    fig_html.write_html(save_path)
    print(f"保存完了: {save_path}")
    print("フォルダを確認して、ブラウザで開いてください。")

樹脂粘度vsせん断速度and流動中圧力(モデルの部分像)

In [ ]:
# セル14

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import plotly.graph_objects as go
import os

# === 設定エリア ===
FIXED_TEMP = temp_med  # デフォルトは中央値 (単位: ℃)
# ==================

if learning_completed:
    # 1. グリッド作成
    n_grid = 50
    sr_mesh = np.geomspace(sr_min, sr_max, n_grid)
    pres_mesh = np.linspace(pres_min, pres_max, n_grid)
    
    X_grid, Y_grid = np.meshgrid(sr_mesh, pres_mesh) # X:SR, Y:Pres
    
    # 2. モデル入力作成
    log_T_in = np.full_like(X_grid.ravel(), np.log(FIXED_TEMP + 273.15)) # 固定
    log_P_in = np.log(Y_grid.ravel())
    log_SR_in = np.log(X_grid.ravel())
    
    X_input_raw = np.stack([log_T_in, log_P_in, log_SR_in], axis=1)
    X_input_scaled = scaler.transform(X_input_raw)
    
    # 3. 予測
    y_pred_scaled = gp_model.predict(X_input_scaled)
    visc_pred = np.exp(y_pred_scaled + y_mean).reshape(X_grid.shape)
    
    # --- A. Matplotlib描画 ---
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(np.log10(X_grid), Y_grid, np.log10(visc_pred), 
                           cmap=cm.plasma, edgecolor='none', alpha=0.9)
    
    ax.set_title(f"3D Map: せん断速度 vs 圧力 (温度={FIXED_TEMP:.1f} ℃ 固定)", fontsize=14)
    ax.set_xlabel('log10(せん断速度 [1/s])', fontsize=12)
    ax.set_ylabel('圧力 [Pa]', fontsize=12)
    ax.set_zlabel('log10(樹脂粘度 [Pa·s])', fontsize=12)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='log10(粘度)')
    plt.show()

    # --- B. Plotly保存 ---
    print("\n>> HTML形式の3Dグラフを作成・保存中...")
    
    base_dir = os.path.dirname(csv_file_path)
    file_name = "3D_Graph_SR_vs_Pres.html"
    save_path = os.path.join(base_dir, file_name)
    
    fig_html = go.Figure(data=[go.Surface(
        z=np.log10(visc_pred), 
        x=np.log10(X_grid), 
        y=Y_grid,
        colorscale='Plasma',
        colorbar=dict(title='log10(粘度)')
    )])
    
    fig_html.update_layout(
        title=f"3D Map: せん断速度 vs 圧力 (温度={FIXED_TEMP:.1f} ℃ 固定)",
        scene=dict(
            xaxis_title='log10(せん断速度 [1/s])',
            yaxis_title='圧力 [Pa]',
            zaxis_title='log10(樹脂粘度 [Pa·s])'
        ),
        width=1000, height=800
    )
    
    fig_html.write_html(save_path)
    print(f"保存完了: {save_path}")

樹脂粘度vs流動中温度and流動中圧力(モデルの部分像)

In [ ]:
# セル15

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import plotly.graph_objects as go
import os

# === 設定エリア ===
FIXED_SR = sr_med  # デフォルトは中央値 (単位: 1/s)
# ==================

if learning_completed:
    # 1. グリッド作成
    n_grid = 50
    temp_mesh = np.linspace(temp_min, temp_max, n_grid)
    pres_mesh = np.linspace(pres_min, pres_max, n_grid)
    
    X_grid, Y_grid = np.meshgrid(temp_mesh, pres_mesh) # X:Temp, Y:Pres
    
    # 2. モデル入力作成
    log_T_in = np.log(X_grid.ravel() + 273.15)
    log_P_in = np.log(Y_grid.ravel())
    log_SR_in = np.full_like(log_T_in, np.log(FIXED_SR)) # 固定
    
    X_input_raw = np.stack([log_T_in, log_P_in, log_SR_in], axis=1)
    X_input_scaled = scaler.transform(X_input_raw)
    
    # 3. 予測
    y_pred_scaled = gp_model.predict(X_input_scaled)
    visc_pred = np.exp(y_pred_scaled + y_mean).reshape(X_grid.shape)
    
    # --- A. Matplotlib描画 ---
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X_grid, Y_grid, np.log10(visc_pred), 
                           cmap=cm.coolwarm, edgecolor='none', alpha=0.9)
    
    ax.set_title(f"3D Map: 温度 vs 圧力 (せん断速度={FIXED_SR:.1f} 1/s 固定)", fontsize=14)
    ax.set_xlabel('温度 [℃]', fontsize=12)
    ax.set_ylabel('圧力 [Pa]', fontsize=12)
    ax.set_zlabel('log10(樹脂粘度 [Pa·s])', fontsize=12)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='log10(粘度)')
    plt.show()

    # --- B. Plotly保存 ---
    print("\n>> HTML形式の3Dグラフを作成・保存中...")
    
    base_dir = os.path.dirname(csv_file_path)
    file_name = "3D_Graph_Temp_vs_Pres.html"
    save_path = os.path.join(base_dir, file_name)
    
    fig_html = go.Figure(data=[go.Surface(
        z=np.log10(visc_pred), 
        x=X_grid, # 温度は線形スケールで見ることが多い
        y=Y_grid,
        colorscale='RdBu',
        colorbar=dict(title='log10(粘度)')
    )])
    
    fig_html.update_layout(
        title=f"3D Map: 温度 vs 圧力 (せん断速度={FIXED_SR:.1f} 1/s 固定)",
        scene=dict(
            xaxis_title='温度 [℃]',
            yaxis_title='圧力 [Pa]',
            zaxis_title='log10(樹脂粘度 [Pa·s])'
        ),
        width=1000, height=800
    )
    
    fig_html.write_html(save_path)
    print(f"保存完了: {save_path}")

予測モデル(データファイル)の保存

In [ ]:
# セル16　学習済みモデルの保存
import os
import joblib

if 'gp_model' in locals() and learning_completed:
    # 1. セル2で選択したCSVファイルの「ディレクトリ（フォルダの場所）」を取得
    base_dir = os.path.dirname(csv_file_path)
    
    # 2. 保存するファイル名を決定
    file_name = 'viscosity_model.pkl'
    
    # 3. パスを結合して、保存先のフルパスを作成
    save_path = os.path.join(base_dir, file_name)
    
    # 4. モデルの保存
    joblib.dump(gp_model, save_path)
    
    print("--- モデル保存完了 ---")
    print(f"保存ファイル名: {file_name}")
    print(f"保存先フォルダ: {base_dir}")
    print(f"フルパス: {save_path}")

else:
    print("エラー: モデルが学習されていないため、保存できません。")